# Map or Ritual? Held-out repositories, issue-based tasks

SWE-bench style tasks from merged pull requests that close a GitHub issue in click, flask and attrs. Runs the same probe agent.

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Official quantized competition models:
#   - 'gemma-4-31b-it-qat-w4a16-ct' (default in sample_submission)
#   - 'gemma-4-12b-it-qat-w4a16-ct'
#   - 'gemma-4-e4b-it-qat-w4a16-ct'
#   - 'gemma-4-e2b-it-qat-w4a16-ct'
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# If switching away from 31B, update all agent YAMLs and remove the 31B-shaped sample LoRA adapters
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

In [ ]:
# ===== HELD-OUT DATA (issues): locate the attached output of the builder notebook =====
import os, re, json, glob, time, random, collections, hashlib
_hits = []
for _d in range(1, 7):
    _hits += glob.glob("/kaggle/input/" + "*/" * _d + "heldout_issues_manifest.json")
assert _hits, "Add the output of the notebook 'maporritual-heldout-issues' as Input first"
HROOT = os.path.dirname(sorted(_hits)[0])
manifest = json.load(open(f"{HROOT}/heldout_issues_manifest.json"))
_lb = open(f"{HROOT}/labels_heldout_issues.json", "rb").read()
_tb = open(f"{HROOT}/tasks.jsonl", "rb").read()
assert hashlib.sha256(_lb).hexdigest() == manifest["labels_sha256"], "labels file changed"
assert hashlib.sha256(_tb).hexdigest() == manifest["tasks_sha256"], "tasks file changed"
print("held-out data:", HROOT, "| tasks:", manifest["n_tasks"])
print("tasks sha256:", manifest["tasks_sha256"][:16], "| labels sha256:", manifest["labels_sha256"][:16])

In [ ]:
# ===== HELD-OUT DRY RUN: does the competition graph library read our files? =====
from swegemma import graph as sg
_T = [json.loads(l) for l in open(f"{HROOT}/tasks.jsonl")]
_L = {r["id"]: r for r in json.load(open(f"{HROOT}/labels_heldout_issues.json"))}
def _strip(n):
    return str(n).split(";", 1)[-1].strip()
checks, problems = 0, []
for short in ("click", "flask", "attrs"):
    _ts = [x for x in _T if x["instance_id"].startswith(short + "_")]
    if not _ts:
        print(short, "has no tasks, skipped")
        continue
    t = _ts[0]
    name = f"{short}_{t['base_commit']}"
    for world in ("W1", "W2"):
        gd, ed = f"{HROOT}/{world}/graphs", f"{HROOT}/embeddings"
        raw = json.load(open(f"{gd}/{name}.json"))
        adj = collections.defaultdict(set)
        for e in raw["edges"]:
            adj[e["source"]].add(e["target"])
        try:
            g = sg.get_graph(repo_name=t["repo"], graph_dir=gd,
                             embeddings_dir=ed, base_commit=t["base_commit"])
            ids = {_strip(n): n for n in g.nodes}
            ok_nodes = set(ids) == {n["id"] for n in raw["nodes"]}
            node = max((n for n in adj if len(adj[n]) <= 15), key=lambda n: len(adj[n]))
            nb = {_strip(x) for x in sg.get_neighbor(node=node, graph=g, max_neighbors=20)} - {node}
            ok_nb = nb == adj[node]
            sim = sg.get_similar_nodes(node=node, repo_name=t["repo"], k=5, graph=g,
                                       graph_dir=gd, embeddings_dir=ed,
                                       base_commit=t["base_commit"])
            ok_sim = len(sim) >= 1
            checks += 1
            print(short, world, "| nodes", len(ids), "match", ok_nodes,
                  "| neighbors of", node.split(".")[-1], len(nb), "match", ok_nb,
                  "| similar", len(sim), str(sim[0])[:90])
            if not (ok_nodes and ok_nb and ok_sim):
                problems.append((short, world, ok_nodes, ok_nb, ok_sim))
        except Exception as e:
            problems.append((short, world, f"{type(e).__name__}: {e}"[:300]))
            print(short, world, "ERROR", f"{type(e).__name__}: {e}"[:300])
print("checks:", checks, "| problems:", problems)
_present = len({x["instance_id"].split("_")[0] for x in _T})
print("DRY RUN PASS" if checks == 2 * _present and _present >= 2 and not problems else "DRY RUN FAIL")
assert checks == 2 * _present and _present >= 2 and not problems, "held-out files are not readable by the graph library, stopping before any GPU work"

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

In [ ]:
# ===== PROBE PART 1H: held-out data, worlds, read-only sandbox =====
import os, re, json, glob, time, random, threading, subprocess
import collections
from pathlib import Path

SRCROOT = f"{HROOT}/src"
EMB = f"{HROOT}/embeddings"
WORLDS = {
    "W1": (f"{HROOT}/W1/graphs", EMB),
    "W2": (f"{HROOT}/W2/graphs", EMB),
}
WORLDS["W3"] = WORLDS["W1"]  # placeholder so later cells run unchanged, never used
TASKS = {}
with open(f"{HROOT}/tasks.jsonl") as f:
    for line in f:
        d = json.loads(line)
        TASKS[d["instance_id"]] = d
with open(f"{HROOT}/labels_heldout_issues.json") as f:
    LABELS = {r["id"]: r for r in json.load(f)}
assert len(TASKS) == len(LABELS) >= 30, (len(TASKS), len(LABELS))

def clip(s, n=3000):
    s = str(s)
    if len(s) <= n:
        return s
    return s[:n] + f"\n...[truncated {len(s) - n} chars]"

def strip(n):
    return str(n).split(";", 1)[-1].strip()

ALLOWED = {"grep", "egrep", "fgrep", "rg", "find", "ls", "cat",
           "head", "tail", "sed", "wc", "sort", "uniq", "cut"}
BAD = (">", "`", "$(", ";", "\n")

def run_command(iid, command):
    cmd = str(command).strip()
    if any(b in cmd for b in BAD):
        return "ERROR: only read-only commands allowed (no ; > ` $( )."
    for seg in re.split(r"\|\||&&|\|", cmd):
        toks = seg.strip().split()
        if not toks or toks[0] not in ALLOWED:
            bad = toks[0] if toks else ""
            return (f"ERROR: command not allowed: {bad}. Use grep, "
                    "find, ls, cat, head, tail, sed -n, wc.")
    try:
        p = subprocess.run(cmd, shell=True, cwd=f"{SRCROOT}/{iid}",
                           capture_output=True, text=True, timeout=20)
        out = p.stdout + p.stderr
    except subprocess.TimeoutExpired:
        return "ERROR: command timed out after 20s."
    return clip(out) or "(no output)"

def read_file(iid, path, start_line=None, end_line=None):
    root = os.path.realpath(f"{SRCROOT}/{iid}")
    rel = str(path).strip().lstrip("/")
    for pre in ("workspace/", "testbed/", "repo/"):
        if rel.startswith(pre):
            rel = rel[len(pre):]
    p = os.path.realpath(os.path.join(root, rel))
    if not p.startswith(root + os.sep) or not os.path.isfile(p):
        return f"ERROR: file not found: {path}"
    with open(p, errors="replace") as f:
        lines = f.read().splitlines()
    s = max(1, int(start_line or 1))
    e = int(end_line or s + 149)
    e = min(e, s + 199, len(lines))
    body = "\n".join(f"{i}: {lines[i - 1]}" for i in range(s, e + 1))
    return clip(f"{rel} lines {s}-{e} of {len(lines)}\n{body}", 6000)

print("PART 1H OK | held-out tasks:", len(TASKS), "| labels:", len(LABELS))

In [ ]:
# ===== PROBE PART 2: graph tools (same code path for every world) =====
from collections import OrderedDict
from swegemma import graph as sg

_gcache, _glock = OrderedDict(), threading.Lock()

def world_graph(world, iid):
    key = (world, iid)
    with _glock:
        if key in _gcache:
            _gcache.move_to_end(key)
            return _gcache[key]
    t = TASKS[iid]
    gd, ed = WORLDS[world]
    g = sg.get_graph(repo_name=t["repo"], graph_dir=gd,
                     embeddings_dir=ed, base_commit=t["base_commit"])
    ids = {strip(n): n for n in g.nodes}
    with _glock:
        _gcache[key] = (g, ids)
        while len(_gcache) > 48:
            _gcache.popitem(last=False)
    return g, ids

def resolve_node(ids, q):
    q = strip(q).strip("`() ")
    if q in ids:
        return q, None
    hits = [s for s in ids if s.endswith("." + q)]
    if len(hits) == 1:
        return hits[0], None
    if hits:
        return None, ("Ambiguous name. Candidates: "
                      + ", ".join(sorted(hits)[:10]))
    return None, (f"Symbol not found in graph: {q}. Use "
                  "search_similar_code or grep to find the dotted path.")

def tool_neighbors(world, iid, node):
    g, ids = world_graph(world, iid)
    nid, err = resolve_node(ids, node)
    if err:
        return err
    nb = sg.get_neighbor(node=nid, graph=g, max_neighbors=20)
    names = sorted({strip(x) for x in nb} - {nid})
    return json.dumps({"node": nid, "neighbors": names})

def tool_similar(world, iid, query, k=10):
    t = TASKS[iid]
    gd, ed = WORLDS[world]
    g, ids = world_graph(world, iid)
    try:
        k = max(1, min(int(k or 10), 20))
    except Exception:
        k = 10
    res = sg.get_similar_nodes(node=strip(query), repo_name=t["repo"],
                               k=k, graph=g, graph_dir=gd,
                               embeddings_dir=ed,
                               base_commit=t["base_commit"])
    out = []
    for r in res:
        if isinstance(r, dict):
            sim = r.get("similarity", r.get("score"))
            if isinstance(sim, (int, float)):
                sim = round(float(sim), 4)
            out.append({"node": strip(r.get("node_name")),
                        "similarity": sim})
        else:
            out.append({"node": strip(r)})
    return json.dumps(out)

def tool_subgraph(world, iid, nodes):
    g, ids = world_graph(world, iid)
    if isinstance(nodes, str):
        nodes = [x for x in re.split(r"[,\s]+", nodes) if x]
    good, errs = [], []
    for q in list(nodes)[:15]:
        nid, err = resolve_node(ids, q)
        if nid:
            good.append(nid)
        else:
            errs.append(err)
    sub = g.subgraph([ids[s] for s in good])
    edges = sorted({tuple(sorted((strip(u), strip(v))))
                    for u, v in sub.edges() if u != v})
    return json.dumps({"nodes": good, "edges": edges, "errors": errs})

def clear_graphs():
    with _glock:
        _gcache.clear()

print("PART 2 OK")

In [ ]:
# ===== PROBE PART 2b: one graph call at a time (thread-safety fix) =====
SG_LOCK = threading.RLock()
if not hasattr(sg, "_orig"):
    sg._orig = (sg.get_graph, sg.get_neighbor, sg.get_similar_nodes)

def _locked(f):
    def g(*a, **k):
        with SG_LOCK:
            return f(*a, **k)
    return g

sg.get_graph = _locked(sg._orig[0])
sg.get_neighbor = _locked(sg._orig[1])
sg.get_similar_nodes = _locked(sg._orig[2])

def world_graph(world, iid):
    key = (world, iid)
    with SG_LOCK:
        if key in _gcache:
            _gcache.move_to_end(key)
            return _gcache[key]
        t = TASKS[iid]
        gd, ed = WORLDS[world]
        g = sg.get_graph(repo_name=t["repo"], graph_dir=gd,
                         embeddings_dir=ed,
                         base_commit=t["base_commit"])
        ids = {strip(n): n for n in g.nodes}
        _gcache[key] = (g, ids)
        while len(_gcache) > 48:
            _gcache.popitem(last=False)
        return g, ids

print("PART 2b OK | graph calls are serialized")

In [ ]:
# ===== PROBE PART 3: tool schemas and prompts =====
def fn(name, desc, props, req):
    return {"type": "function", "function": {
        "name": name, "description": desc,
        "parameters": {"type": "object", "properties": props,
                       "required": req}}}

S = {"type": "string"}
I = {"type": "integer"}
L = {"type": "array", "items": {"type": "string"}}
T_RUN = fn("run_command",
           "Run a read-only shell command (grep, find, ls, cat, head, "
           "tail, sed -n, wc) in the repository root.",
           {"command": S}, ["command"])
T_READ = fn("read_file",
            "Read lines of a file. Paths are relative to the "
            "repository root.",
            {"path": S, "start_line": I, "end_line": I}, ["path"])
T_SIM = fn("search_similar_code",
           "Find functions/classes with similar precomputed embeddings "
           "given a symbol name (e.g. pkg.module.Class.method, "
           "Class.method, or func_name).",
           {"query": S, "k": I}, ["query"])
T_NB = fn("get_code_neighbors",
          "List the callers and callees of a symbol in the pre-built "
          "code graph.", {"node": S}, ["node"])
T_SUB = fn("get_code_subgraph",
           "Return the call edges among a list of symbols in the "
           "pre-built code graph.", {"nodes": L}, ["nodes"])
T_SUBMIT = fn("submit_localization",
              "Submit the final answer: up to 5 fully qualified "
              "function or method names (dotted paths such as "
              "pkg.module.Class.method) whose code must change to "
              "resolve the issue, most likely first.",
              {"functions": L}, ["functions"])

BASE_SYS = (
    "You are an expert software engineer localizing a GitHub issue. "
    "The repository {repo} is checked out at the commit where the "
    "issue was reported; the checkout contains its Python source "
    "files. Find the functions or methods whose code must change to "
    "resolve the issue. Investigate with the tools, one tool call at "
    "a time. When you are confident, call submit_localization with "
    "up to 5 fully qualified dotted names (for example "
    "package.module.Class.method), most likely first. You have at "
    "most {budget} tool calls.")
GRAPH_NOTE = (
    "\n\nThis repository has pre-built code graph and embedding data "
    "indexed by dotted symbol path (e.g. pkg.module.Class.method). "
    "Use these tools for fast, targeted navigation: "
    "search_similar_code, get_code_neighbors, get_code_subgraph.")
NR_SYS = (
    "You are an expert software engineer. You cannot see the code. "
    "Using only the issue text and what you know about the "
    "repository {repo}, guess which functions or methods must change "
    "to resolve the issue. Call submit_localization with up to 5 "
    "fully qualified dotted names, most likely first.")
NUDGE = ("Please continue by calling one of the tools. When you are "
         "confident, call submit_localization.")
FORCE = ("Your tool budget is used up. Call submit_localization now "
         "with your best answer.")
GRAPH_TOOLS = {"search_similar_code", "get_code_neighbors",
               "get_code_subgraph"}

def tools_for(world):
    if world == "NR":
        return [T_SUBMIT]
    t = [T_RUN, T_READ]
    if world in WORLDS:
        t += [T_SIM, T_NB, T_SUB]
    return t + [T_SUBMIT]

print("PART 3 OK")

In [ ]:
# ===== PROBE PART 4: model client and episode loop =====
import requests

ROOT = server_instance.base_url.rstrip("/")
ROOT = ROOT[:-3] if ROOT.endswith("/v1") else ROOT
H = {"Authorization": "Bearer EMPTY"}
MID = requests.get(ROOT + "/v1/models", headers=H,
                   timeout=60).json()["data"][0]["id"]
MODEL_TAG = MID.rstrip("/").split("/")[-2]

def chat(messages, tools, seed, temperature=0.7, max_tokens=2048):
    p = {"model": MID, "messages": messages, "tools": tools,
         "tool_choice": "auto", "temperature": temperature,
         "max_tokens": max_tokens, "seed": seed}
    r = requests.post(ROOT + "/v1/chat/completions", json=p,
                      headers=H, timeout=600).json()
    if "choices" not in r:
        raise RuntimeError(str(r)[:300])
    return r

def dispatch(world, iid, name, a):
    if name == "run_command":
        return run_command(iid, a.get("command", ""))
    if name == "read_file":
        return read_file(iid, a.get("path", ""),
                         a.get("start_line"), a.get("end_line"))
    if world in WORLDS and name == "search_similar_code":
        return tool_similar(world, iid, a.get("query", ""),
                            a.get("k", 10))
    if world in WORLDS and name == "get_code_neighbors":
        return tool_neighbors(world, iid, a.get("node", ""))
    if world in WORLDS and name == "get_code_subgraph":
        return tool_subgraph(world, iid, a.get("nodes", []))
    return f"ERROR: unknown tool {name}"

def parse_pred(a):
    fs = a.get("functions", []) if isinstance(a, dict) else []
    if isinstance(fs, str):
        fs = re.split(r"[,\s]+", fs)
    return [strip(x) for x in fs if str(x).strip()][:5]

def run_episode(iid, world, seed, budget=20):
    t = TASKS[iid]
    sysp = (NR_SYS if world == "NR" else BASE_SYS).format(
        repo=t["repo"], budget=budget)
    if world in WORLDS:
        sysp += GRAPH_NOTE
    msgs = [{"role": "system", "content": sysp},
            {"role": "user",
             "content": "Issue:\n" + t["problem_statement"][:6000]}]
    log = {"id": iid, "world": world, "seed": seed,
           "model": MODEL_TAG, "budget": budget, "steps": [],
           "pred": [], "status": "", "nudges": 0,
           "max_prompt_tokens": 0, "completion_tokens": 0}
    T0, calls, forced = time.time(), 0, False
    tools = tools_for(world)
    while True:
        if (calls >= budget or log["max_prompt_tokens"] > 26000) \
                and not forced:
            forced = True
            tools = [T_SUBMIT]
            msgs.append({"role": "user", "content": FORCE})
        try:
            r = chat(msgs, tools, seed * 1000 + calls + log["nudges"])
        except Exception as e:
            log["status"], log["error"] = "API_ERROR", str(e)[:300]
            break
        u = r.get("usage", {})
        log["max_prompt_tokens"] = max(log["max_prompt_tokens"],
                                       u.get("prompt_tokens", 0))
        log["completion_tokens"] += u.get("completion_tokens", 0)
        m = r["choices"][0]["message"]
        tcs = m.get("tool_calls") or []
        if not tcs:
            if log["nudges"] >= 2 or forced:
                log["status"] = "NO_SUBMIT"
                break
            log["nudges"] += 1
            msgs.append({"role": "assistant",
                         "content": m.get("content") or ""})
            msgs.append({"role": "user", "content": NUDGE})
            continue
        tc = tcs[0]
        name = tc["function"]["name"]
        try:
            a = json.loads(tc["function"]["arguments"] or "{}")
        except Exception:
            a = None
        msgs.append({"role": "assistant",
                     "content": m.get("content") or "",
                     "tool_calls": [tc]})
        if a is not None and name == "submit_localization":
            log["pred"] = parse_pred(a)
            log["status"] = "SUBMIT_FORCED" if forced else "SUBMIT"
            break
        calls += 1
        if a is None:
            out = "ERROR: tool arguments were not valid JSON."
        elif forced:
            out = "ERROR: only submit_localization is allowed now."
        else:
            try:
                out = dispatch(world, iid, name, a)
            except Exception as e:
                out = f"ERROR: {type(e).__name__}: {str(e)[:200]}"
        log["steps"].append({"tool": name, "args": a, "out": out,
                             "t": round(time.time() - T0, 1)})
        msgs.append({"role": "tool", "name": name,
                     "tool_call_id": tc.get("id", f"call{calls}"),
                     "content": out})
        if forced:
            log["forced_tries"] = log.get("forced_tries", 0) + 1
            if log["forced_tries"] >= 2:
                log["status"] = "NO_SUBMIT"
                break
    log["calls"] = calls
    log["seconds"] = round(time.time() - T0, 1)
    return log

print("PART 4 OK | model:", MODEL_TAG)

In [ ]:
# ===== PROBE PART 4b: shorter request timeout =====
def chat(messages, tools, seed, temperature=0.7, max_tokens=2048):
    p = {"model": MID, "messages": messages, "tools": tools,
         "tool_choice": "auto", "temperature": temperature,
         "max_tokens": max_tokens, "seed": seed}
    r = requests.post(ROOT + "/v1/chat/completions", json=p,
                      headers=H, timeout=400).json()
    if "choices" not in r:
        raise RuntimeError(str(r)[:300])
    return r

print("PART 4b OK | timeout 180 s")

In [ ]:
# ===== PROBE PART 5: scoring, batch runner, summary =====
import concurrent.futures as cf

_files = {}
def py_files(iid):
    if iid not in _files:
        root = f"{SRCROOT}/{iid}"
        out = []
        for d, _, fs in os.walk(root):
            out += [os.path.relpath(f"{d}/{x}", root)
                    for x in fs if x.endswith(".py")]
        _files[iid] = out
    return _files[iid]

def resolve_file(iid, nid):
    parts = nid.split(".")
    for i in range(len(parts), 0, -1):
        mod = "/".join(parts[:i])
        hits = [f for f in py_files(iid)
                if f == mod + ".py" or f.endswith("/" + mod + ".py")
                or f == mod + "/__init__.py"
                or f.endswith("/" + mod + "/__init__.py")]
        if hits:
            return min(hits, key=len)
    return None

def norm(p):
    p = strip(p).strip("`'\"() ").replace("::", ".")
    p = p.replace(".py.", ".").replace("/", ".")
    return p[:-3] if p.endswith(".py") else p

def match(pred, gold):
    p = norm(pred)
    if not p:
        return False
    if p == gold or p.endswith("." + gold):
        return True
    return "." in p and gold.endswith("." + p)

def score(log):
    lab = LABELS[log["id"]]
    gold, preds = lab["gold"], log["pred"]
    log["hit5"] = any(match(p, g) for p in preds for g in gold)
    log["hit1"] = bool(preds) and any(match(preds[0], g) for g in gold)
    gf = {resolve_file(log["id"], g) for g in gold} - {None}
    pf = {resolve_file(log["id"], norm(p)) for p in preds} - {None}
    log["file_hit5"] = bool(gf & pf)
    st = log["steps"]
    log["graph_calls"] = sum(s["tool"] in GRAPH_TOOLS for s in st)
    seen = [i for i, s in enumerate(st) if s["tool"] in GRAPH_TOOLS
            and any(g in str(s["out"]) for g in gold)]
    log["graph_exposed"] = bool(seen)
    log["exposed_then_hit"] = bool(seen) and log["hit5"]
    log["primary"] = lab["primary"]
    log["hop"] = lab["hop"]
    log["n_anchors"] = lab["n_anchors"]
    return log

def safe_episode(iid, world, seed, budget):
    try:
        return run_episode(iid, world, seed, budget)
    except Exception as e:
        return {"id": iid, "world": world, "seed": seed,
                "model": MODEL_TAG, "budget": budget, "steps": [],
                "pred": [], "status": "CRASH", "calls": 0,
                "seconds": 0, "error": f"{type(e).__name__}: {e}"[:300]}

def run_batch(ids, worlds, seeds, out_path, budget=20, workers=24):
    done = set()
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                d = json.loads(line)
                if d["status"] not in ("CRASH", "API_ERROR"):
                    done.add((d["id"], d["world"], d["seed"]))
    jobs = [(i, w, s) for i in ids for w in worlds for s in seeds
            if (i, w, s) not in done]
    print("jobs to run:", len(jobs), "| already done:", len(done))
    T0, n = time.time(), 0
    with cf.ThreadPoolExecutor(workers) as ex, \
            open(out_path, "a") as f:
        futs = [ex.submit(safe_episode, *j, budget) for j in jobs]
        for fu in cf.as_completed(futs):
            r = fu.result()
            log = r if r["status"] == "CRASH" else score(r)
            f.write(json.dumps(log) + "\n")
            f.flush()
            n += 1
            if n % 20 == 0 or n == len(jobs):
                print(f"  {n}/{len(jobs)} | {round(time.time() - T0)} s")
    clear_graphs()

def load_runs(path):
    with open(path) as f:
        rows = [json.loads(x) for x in f]
    last = {}
    for r in rows:
        last[(r["id"], r["world"], r["seed"])] = r
    return list(last.values())

def summary(path):
    rs = load_runs(path)
    by = collections.defaultdict(list)
    for r in rs:
        by[r["world"]].append(r)
    print(f"{'world':6}{'n':>5}{'hit@5':>7}{'hit@1':>7}{'file@5':>8}"
          f"{'calls':>7}{'graph%':>8}{'exposed':>9}{'sec':>6}")
    for w in sorted(by):
        x = by[w]
        n = len(x)
        mean = lambda k: sum(float(r.get(k) or 0) for r in x) / n
        calls = sum(r.get("calls", 0) for r in x)
        gshare = sum(r.get("graph_calls", 0) for r in x) / max(calls, 1)
        print(f"{w:6}{n:>5}{mean('hit5'):>7.2f}{mean('hit1'):>7.2f}"
              f"{mean('file_hit5'):>8.2f}{mean('calls'):>7.1f}"
              f"{gshare:>8.2f}{mean('graph_exposed'):>9.2f}"
              f"{mean('seconds'):>6.0f}")
    st = collections.Counter((r["world"], r["status"]) for r in rs)
    print("status:", sorted(st.items()))
    errs = [r.get("error") for r in rs if r.get("error")]
    if errs:
        print("example error:", errs[0])

print("PART 5 OK")

In [ ]:
# ===== PROBE PART 7: graph-first arm (patch only, safe to re-run) =====
import inspect
GRAPH_FIRST = (
    "\n\nRequired workflow: first call search_similar_code on symbol "
    "names from the issue, then call get_code_neighbors on the most "
    "relevant results to follow callers and callees, and only then "
    "read files to confirm. Base your answer on what the graph shows.")
for w in ("W1", "W2", "W3"):
    WORLDS[w + "G"] = WORLDS[w]
if "GRAPH_FIRST" in run_episode.__code__.co_names:
    print("run_episode already patched, skipping")
else:
    _src = inspect.getsource(run_episode)
    _old = "        sysp += GRAPH_NOTE\n"
    _new = ("        sysp += GRAPH_NOTE + (GRAPH_FIRST if "
            "world.endswith('G') else '')\n")
    assert _old in _src, "patch point not found"
    exec(_src.replace(_old, _new))
assert "GRAPH_FIRST" in run_episode.__code__.co_names
print("PART 7 OK | worlds:", sorted(WORLDS))

In [ ]:
# ===== PROBE PART H: held-out repositories (click, flask, attrs) =====
HELD = sorted(LABELS)
assert all(i.split("_")[0] in ("click", "flask", "attrs") for i in HELD)
print("model:", MODEL_TAG, "| held-out tasks:", len(HELD))
ORDER_T = list(HELD)
random.Random(20261116).shuffle(ORDER_T)
T_WORLDS = ["W0", "W1", "W2", "W1G", "W2G"]
T_SEEDS = [1, 2]
PER_TASK = len(T_WORLDS) * len(T_SEEDS)
OUT_T = f"/kaggle/working/heldout_issues_{MODEL_TAG}.jsonl"
WORKERS_T = 16 if "31b" in MODEL_TAG else 24
LIMIT_H, CHUNK, MAX_RESTARTS = 6.0, 4, 8
T_BEGIN = time.time()
restarts = 0

def alive():
    try:
        r = requests.get(ROOT + "/v1/models", headers=H, timeout=20)
        return r.status_code == 200
    except Exception:
        return False

def restart_server():
    global server_instance
    try:
        server_instance.stop()
    except Exception as e:
        print("stop failed:", type(e).__name__)
    time.sleep(30)
    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
    server_instance.start()
    print("server restarted | alive:", alive())

def n_ok(chunk):
    if not os.path.exists(OUT_T):
        return 0
    return sum(1 for r in load_runs(OUT_T) if r["id"] in chunk
               and r["status"] not in ("API_ERROR", "CRASH"))

print("episodes planned:", len(ORDER_T) * PER_TASK,
      "| workers:", WORKERS_T)
finished = 0
for k in range(0, len(ORDER_T), CHUNK):
    hours = (time.time() - T_BEGIN) / 3600
    if hours > LIMIT_H:
        print("TIME GUARD: stopping at", round(hours, 2), "h")
        break
    if restarts > MAX_RESTARTS:
        print("TOO MANY RESTARTS: stopping")
        break
    chunk = ORDER_T[k:k + CHUNK]
    for attempt in range(3):
        if not alive():
            restarts += 1
            print("server dead, restart", restarts)
            try:
                restart_server()
            except Exception as e:
                print("restart failed:", type(e).__name__, str(e)[:200])
                break
        run_batch(chunk, T_WORLDS, T_SEEDS, OUT_T,
                  budget=20, workers=WORKERS_T)
        if n_ok(chunk) == len(chunk) * PER_TASK:
            break
    finished += len(chunk)
    print("== chunk done | ok episodes:", n_ok(chunk), "/",
          len(chunk) * PER_TASK, "| tasks:", finished, "/", len(ORDER_T),
          "| hours:", round((time.time() - T_BEGIN) / 3600, 2),
          "| restarts:", restarts)
summary(OUT_T)
try:
    server_instance.stop()
except Exception:
    pass
print("PART H DONE | tasks attempted:", finished, "| restarts:", restarts)